# Домашнее задание: Построение и анализ филогенетических деревьев

**Ген:** Цитохром c (CYCS)

## Установка зависимостей

In [1]:
!pip install biopython -q
!apt-get install -y mafft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  fonts-lato libauthen-sasl-perl libclone-perl libdata-dump-perl
  libencode-locale-perl libfile-listing-perl libfont-afm-perl
  libhtml-form-perl libhtml-format-perl libhtml-parser-perl
  libhtml-tagset-perl libhtml-tree-perl libhttp-cookies-perl
  libhttp-daemon-perl libhttp-date-perl libhttp-message-perl
  libhttp-negotiate-perl libio-html-perl libio-socket-ssl-perl
  liblwp-mediatypes-perl liblwp-protocol-https-perl libmailtools-perl
  libnet-http-perl libnet-smtp-ssl-perl libnet-ssleay-perl libruby3.0
  libtry-tiny-perl liburi-perl libwww-perl libwww-robotrules-perl lynx
  lynx-common netbase perl-openssl-defaults rake ruby ruby-net-telnet
  ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration
Suggested packages:
  libdigest-hmac-perl libgssapi-perl libcrypt

## Задание 1. Подготовка данных

In [2]:
# Загрузка файла с локального компьютера
from google.colab import files
uploaded = files.upload()  # выбери sequence.fasta

Saving sequence.fasta to sequence.fasta


In [3]:
# Копируем и обрезаем до 100 последовательностей
import shutil, os
from Bio import SeqIO

shutil.copy(list(uploaded.keys())[0], "cycs_raw_full.fasta")

all_records = list(SeqIO.parse("cycs_raw_full.fasta", "fasta"))
print(f"Всего последовательностей: {len(all_records)}")

subset = all_records[:100]
SeqIO.write(subset, "cycs_raw.fasta", "fasta")
print(f"Оставлено для работы: {len(subset)}")
print(f"Размер файла: {os.path.getsize('cycs_raw.fasta')} байт")

Всего последовательностей: 583
Оставлено для работы: 100
Размер файла: 16409 байт


In [4]:
# 1.3 -- переименование заголовков
# Было:  >NP_061820.1 CYCS [organism=Homo sapiens] [GeneID=54205]
# Стало: >Homo_sapiens_NP_061820.1

import re

renamed_records = []

for rec in SeqIO.parse("cycs_raw.fasta", "fasta"):
    accession = rec.id

    match = re.search(r'\[organism=([^\]]+)\]', rec.description)
    if match:
        organism = match.group(1).replace(" ", "_")
    else:
        # Если тег organism отсутствует -- берём первые два слова описания
        words = rec.description.split()
        organism = "_".join(words[1:3]) if len(words) >= 3 else "Unknown"

    rec.id = f"{organism}_{accession}"
    rec.description = ""
    rec.name = ""
    renamed_records.append(rec)

SeqIO.write(renamed_records, "cycs_renamed.fasta", "fasta")
print(f"Переименовано: {len(renamed_records)} записей")

# Проверка первых 5 заголовков
print("\nПервые 5 заголовков:")
for rec in renamed_records[:5]:
    print(rec.id)

Переименовано: 100 записей

Первые 5 заголовков:
cytochrome_c_XP_007466128.1
cytochrome_c_XP_039405800.1
cytochrome_c_XP_010174923.1
cytochrome_c_XP_082702755.1
cytochrome_c_XP_082702754.1


## Задание 2. Множественное выравнивание (MAFFT)

In [5]:
import subprocess

result = subprocess.run(
    ["mafft", "--auto", "--thread", "-1", "cycs_renamed.fasta"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    with open("cycs_aligned.fasta", "w") as f:
        f.write(result.stdout)
    print("Выравнивание готово: cycs_aligned.fasta")
else:
    print("Ошибка MAFFT:")
    print(result.stderr[:500])

Выравнивание готово: cycs_aligned.fasta


In [6]:
# Проверка выравнивания
from Bio import AlignIO

alignment = AlignIO.read("cycs_aligned.fasta", "fasta")
total_chars = len(alignment) * alignment.get_alignment_length()
gap_count = sum(str(rec.seq).count('-') for rec in alignment)

print(f"Последовательностей: {len(alignment)}")
print(f"Длина выравнивания:   {alignment.get_alignment_length()} позиций")
print(f"Доля гэпов:           {gap_count / total_chars:.2%}")

Последовательностей: 100
Длина выравнивания:   185 позиций
Доля гэпов:           42.62%


In [7]:
# Скачать готовое выравнивание на компьютер
from google.colab import files
files.download("cycs_aligned.fasta")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Проверить первые 5 заголовков переименованного файла
from Bio import SeqIO
for rec in list(SeqIO.parse("cycs_renamed.fasta", "fasta"))[:5]:
    print(rec.id)

cytochrome_c_XP_007466128.1
cytochrome_c_XP_039405800.1
cytochrome_c_XP_010174923.1
cytochrome_c_XP_082702755.1
cytochrome_c_XP_082702754.1


In [9]:
import re
from Bio import SeqIO

renamed_records = []

for rec in SeqIO.parse("cycs_raw.fasta", "fasta"):
    accession = rec.id
    description = rec.description

    # Пробуем вытащить organism из скобок
    match = re.search(r'\[([A-Z][a-z]+ [a-z]+)\]', description)
    if match:
        organism = match.group(1).replace(" ", "_")
    else:
        # Берём последние слова в скобках -- там обычно вид
        words = description.split("[")
        candidates = [w.split("]")[0] for w in words[1:] if w.split("]")[0].count(" ") == 1]
        if candidates:
            organism = candidates[0].replace(" ", "_")
        else:
            organism = "Unknown"

    rec.id = f"{organism}_{accession}"
    rec.description = ""
    rec.name = ""
    renamed_records.append(rec)

SeqIO.write(renamed_records, "cycs_renamed.fasta", "fasta")

# Проверка
for rec in renamed_records[:5]:
    print(rec.id)

Lipotes_vexillifer_XP_007466128.1
Unknown_XP_039405800.1
Antrostomus_carolinensis_XP_010174923.1
Leucopleurus_acutus_XP_082702755.1
Leucopleurus_acutus_XP_082702754.1


In [10]:
from Bio import SeqIO
# Посмотрим на сырое описание проблемных записей
for rec in list(SeqIO.parse("cycs_raw.fasta", "fasta"))[:10]:
    print(rec.description)

XP_007466128.1 cytochrome c [Lipotes vexillifer]
XP_039405800.1 cytochrome c [Corvus cornix cornix]
XP_010174923.1 cytochrome c [Antrostomus carolinensis]
XP_082702755.1 cytochrome c isoform 1 [Leucopleurus acutus]
XP_082702754.1 cytochrome c isoform 1 [Leucopleurus acutus]
XP_082702753.1 cytochrome c isoform 1 [Leucopleurus acutus]
XP_082549911.1 cytochrome c [Mustela nivalis]
XP_082488307.1 cytochrome c isoform 1 [Mesoplodon mirus]
XP_082488306.1 cytochrome c isoform 1 [Mesoplodon mirus]
XP_082488305.1 cytochrome c isoform 1 [Mesoplodon mirus]


In [11]:
import re
from Bio import SeqIO

renamed_records = []

for rec in SeqIO.parse("cycs_raw.fasta", "fasta"):
    accession = rec.id
    description = rec.description

    # Берём всё что внутри последних квадратных скобок
    match = re.search(r'\[([^\[\]]+)\]$', description.strip())
    if match:
        organism = match.group(1).replace(" ", "_")
    else:
        organism = "Unknown"

    rec.id = f"{organism}_{accession}"
    rec.description = ""
    rec.name = ""
    renamed_records.append(rec)

SeqIO.write(renamed_records, "cycs_renamed.fasta", "fasta")

for rec in renamed_records[:10]:
    print(rec.id)

Lipotes_vexillifer_XP_007466128.1
Corvus_cornix_cornix_XP_039405800.1
Antrostomus_carolinensis_XP_010174923.1
Leucopleurus_acutus_XP_082702755.1
Leucopleurus_acutus_XP_082702754.1
Leucopleurus_acutus_XP_082702753.1
Mustela_nivalis_XP_082549911.1
Mesoplodon_mirus_XP_082488307.1
Mesoplodon_mirus_XP_082488306.1
Mesoplodon_mirus_XP_082488305.1


In [12]:
import subprocess

result = subprocess.run(
    ["mafft", "--auto", "--thread", "-1", "cycs_renamed.fasta"],
    capture_output=True,
    text=True
)

with open("cycs_aligned.fasta", "w") as f:
    f.write(result.stdout)

print("Готово!")

from google.colab import files
files.download("cycs_aligned.fasta")

Готово!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>